# Mental Health Support Chatbot Fine Tuning

This project fine tunes a language model on empathetic dialogue data to generate supportive and context aware responses.

# Environment Setup

Install the required libraries for dataset processing, model training, and inference.

In [1]:
!pip install transformers datasets torch accelerate -q

# Dataset Loading

Load the EmpatheticDialogues dataset that contains conversations focused on emotions and empathy.

In [3]:
from datasets import load_dataset

# Load Facebook AI's empathetic dialogues dataset
dataset = load_dataset('Ahren09/empathetic_dialogues')
print(dataset)
print(dataset['train'][0])

README.md:   0%|          | 0.00/757 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.76M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/604k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/608k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6340 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5714 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 84169
    })
    validation: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 6340
    })
    test: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 5714
    })
})
{'conv_id': 'hit:0_conv:1', 'utterance_idx': '1', 'context': 'sentimental', 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'speaker_idx': '1', 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'selfeval': '5|5|5_2|

# Model Initialization

Load the DistilGPT2 model and tokenizer that will be fine tuned for the chatbot.

In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a lightweight pre-trained GPT-2 model and its tokenizer
model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token to EOS because GPT-2 does not define a pad token by default
tokenizer.pad_token = tokenizer.eos_token

# Load the actual language model that will be fine-tuned
model = AutoModelForCausalLM.from_pretrained(model_name)

print('Model loaded!')

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded!


# Data Preprocessing

Prepare and tokenize the dialogue data into a format suitable for model training.

In [37]:
def preprocess(example):
  text = f"User: {example['prompt']}\nTherapist: {example['utterance']}"

  # Tokenize text into numbers the model understands
  return tokenizer(text, truncation=True, max_length=128, padding='max_length')

# Select only 1000 samples to make training faster and lightweight
small_train = dataset['train'].select(range(5000))

# Apply preprocessing to each example in dataset
tokenized = small_train.map(preprocess, remove_columns=small_train.column_names)

# Add labels (target output for training)
tokenized = tokenized.add_column('labels', tokenized['input_ids'])
print('Dataset prepared!')

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset prepared!


# Model Training

Configure the training process and fine tune the model on the prepared dataset.

In [38]:
from transformers import TrainingArguments, Trainer

# Define training configuration such as epochs, batch size, learning rate, and logging settings
training_args = TrainingArguments(
  output_dir='./MindCare_Model',
  num_train_epochs=2,
  per_device_train_batch_size=8,
  save_steps=500,
  logging_steps=100,
  learning_rate=5e-5,
  fp16=True, # enables faster training using mixed precision on GPU
  report_to='none' # disables external logging tools like WandB
)

# Initialize Trainer which handles the training loop automatically
trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=tokenized,
)

# Start the fine-tuning process
trainer.train()
print("Model fine-tuning finished. Ready for inference.")

{'loss': '1.908', 'grad_norm': '2.694', 'learning_rate': '4.604e-05', 'epoch': '0.16'}
{'loss': '1.075', 'grad_norm': '2.187', 'learning_rate': '4.204e-05', 'epoch': '0.32'}
{'loss': '1.047', 'grad_norm': '1.954', 'learning_rate': '3.804e-05', 'epoch': '0.48'}
{'loss': '1.044', 'grad_norm': '1.594', 'learning_rate': '3.404e-05', 'epoch': '0.64'}
{'loss': '0.9877', 'grad_norm': '1.489', 'learning_rate': '3.004e-05', 'epoch': '0.8'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.9734', 'grad_norm': '1.647', 'learning_rate': '2.604e-05', 'epoch': '0.96'}
{'loss': '0.9288', 'grad_norm': '1.513', 'learning_rate': '2.204e-05', 'epoch': '1.12'}
{'loss': '0.9302', 'grad_norm': '1.693', 'learning_rate': '1.804e-05', 'epoch': '1.28'}
{'loss': '0.9128', 'grad_norm': '1.754', 'learning_rate': '1.404e-05', 'epoch': '1.44'}
{'loss': '0.8963', 'grad_norm': '1.687', 'learning_rate': '1.004e-05', 'epoch': '1.6'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.9117', 'grad_norm': '1.748', 'learning_rate': '6.04e-06', 'epoch': '1.76'}
{'loss': '0.8839', 'grad_norm': '1.635', 'learning_rate': '2.04e-06', 'epoch': '1.92'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '160.6', 'train_samples_per_second': '62.28', 'train_steps_per_second': '7.785', 'train_loss': '1.035', 'epoch': '2'}
Model fine-tuning finished. Ready for inference.


# Chatbot Evaluation

Generate responses for sample mental health related prompts to evaluate the model.

In [48]:
import warnings
import transformers
from transformers import pipeline

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

model.config.max_length = None
model.generation_config.max_length = None

# Create a text generation pipeline using your fine-tuned model
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def mental_health_response(user_message):
  # Format input like a conversation prompt
  prompt = f'User: {user_message}\nTherapist:'

  result = chatbot(
      prompt,
      max_new_tokens=60,
      do_sample=True,
      max_length=None,
      temperature=0.5,
      top_k=50,
      top_p=0.92,
      no_repeat_ngram_size=2,
      pad_token_id=tokenizer.eos_token_id
  )

  # Full generated text returned by model
  generated = result[0]['generated_text']

  # Extract only the part after "Therapist:"
  return generated.split('Therapist:')[-1].strip()

# Test inputs to check chatbot behavior
test_inputs = [
    "I feel overwhelmed with everything going on in my life.",
    "I keep overthinking small things and it’s exhausting."
]

for msg in test_inputs:
  print(f'User: {msg}')
  print(f'Bot: {mental_health_response(msg)}')
  print('-' * 60)

User: I feel overwhelmed with everything going on in my life.
Bot: It's been a lot of work and I'm grateful for everything.
------------------------------------------------------------
User: I keep overthinking small things and it’s exhausting.
Bot: That is so frustrating.
------------------------------------------------------------


# Mental Health Chatbot Fine Tuning Pipeline Summary

* Installed transformers, accelerate, and datasets libraries to configure the cloud GPU environment
* Downloaded the empathetic dialogues dataset from Hugging Face to get emotional conversational data
* Initialized the pre trained distilgpt2 model as a clean slate baseline for fine tuning
* Mapped the padding token to the end of sentence token to fix default GPT2 configurations
* Corrected the data pairing typo by mapping user prompt to user and utterance to therapist
* Expanded the training dataset sample range from 1000 to 5000 rows to improve model learning
* Tokenized the conversations while applying truncation and padding at a maximum length of 128 tokens
* Configured the training arguments with a batch size of 8 over 2 full training epochs
* Enabled mixed precision training via fp16 to leverage the free cloud T4 GPU acceleration
* Utilized the python warnings and transformers logging libraries to permanently silence output terminal errors
* Optimized generation parameters by locking the inference temperature to 0.5 to balance logic and variety
* Implemented top k, top p, and no repeat ngram size settings to physically eliminate repetitive bot responses